In [26]:
import os
import cv2
import numpy as np
import time as time_module
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

In [27]:
# =============================
# CONFIG
# =============================
IMAGE_DIR = "images"
OUTPUT_DIR = "output"

SAM_CHECKPOINT = "sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = "cuda"   # change to "cuda" if available

# Object-level tuning

# fewer objects	Increase MIN_MASK_AREA
# More objects	Decrease MIN_MASK_AREA
# Aggressive merge	Lower OVERLAP_IOU_THRESH
# Faster	POINTS_PER_SIDE = 8
# Cleaner	Increase STABILITY_THRESH
POINTS_PER_SIDE = 8
MIN_MASK_AREA = 20000
PRED_IOU_THRESH = 0.9
STABILITY_THRESH = 0.97
OVERLAP_IOU_THRESH = 0.10

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================
# Load SAM
# =============================
sam = sam_model_registry[MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=DEVICE)

mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=POINTS_PER_SIDE,
    pred_iou_thresh=PRED_IOU_THRESH,
    stability_score_thresh=STABILITY_THRESH,
    min_mask_region_area=MIN_MASK_AREA,
)

# =============================
# IoU function
# =============================
def mask_iou(mask1, mask2):
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return intersection / union if union > 0 else 0

In [28]:
# =============================
# Process images
# =============================
start_time = 0
end_time = 0
time = 0
for image_file in os.listdir(IMAGE_DIR):
    start_time = time_module.time()
    if not image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(IMAGE_DIR, image_file)
    image_name = os.path.splitext(image_file)[0]

    print(f"\nProcessing: {image_file}")

    out_dir = os.path.join(OUTPUT_DIR, image_name)
    os.makedirs(out_dir, exist_ok=True)

    # Load image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Generate masks
    masks = mask_generator.generate(image)

    print(f"Initial masks: {len(masks)}")

    # Sort masks by area (largest first)
    masks = sorted(masks, key=lambda x: x["area"], reverse=True)

    # =============================
    # Object-level filtering
    # =============================
    final_masks = []

    for mask_data in masks:
        mask = mask_data["segmentation"]

        keep = True
        for kept in final_masks:
            iou = mask_iou(mask, kept["segmentation"])
            if iou > OVERLAP_IOU_THRESH:
                keep = False
                break

        if keep:
            final_masks.append(mask_data)

    print(f"Object-level masks: {len(final_masks)}")

    # =============================
    # Save cropped segments
    # =============================
    for i, mask_data in enumerate(final_masks):
        mask = mask_data["segmentation"]

        segmented = np.zeros_like(image)
        segmented[mask] = image[mask]

        x, y, w, h = mask_data["bbox"]
        crop = segmented[int(y):int(y+h), int(x):int(x+w)]

        crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)

        out_path = os.path.join(out_dir, f"object_{i}.png")
        cv2.imwrite(out_path, crop_bgr)

    print(f"Saved to: {out_dir}")
    #execution tyme in real time

    
    end_time = time_module.time()
    time = end_time - start_time

    print(f"time to process: {time}")

print("\n✅ All images processed successfully.")


Processing: bhaat_002.jpg
Initial masks: 6
Object-level masks: 5
Saved to: output\bhaat_002
time to process: 14.89491891860962

Processing: bhaat_003.jpg
Initial masks: 6
Object-level masks: 6
Saved to: output\bhaat_003
time to process: 14.750788927078247

Processing: bhaat_004.jpg
Initial masks: 5
Object-level masks: 2
Saved to: output\bhaat_004
time to process: 15.545244455337524

Processing: bhaat_005.jpg
Initial masks: 8
Object-level masks: 6
Saved to: output\bhaat_005
time to process: 14.984467506408691

Processing: boiled_egg_001.jpg
Initial masks: 19
Object-level masks: 10
Saved to: output\boiled_egg_001
time to process: 15.676503419876099

Processing: boiled_egg_002.jpg
Initial masks: 11
Object-level masks: 11
Saved to: output\boiled_egg_002
time to process: 15.958412885665894

Processing: boiled_egg_003.jpg
Initial masks: 9
Object-level masks: 7
Saved to: output\boiled_egg_003
time to process: 15.710431337356567

Processing: boiled_egg_004.jpg
Initial masks: 10
Object-level m